In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torchxrayvision as xrv
import torchvision.transforms as T
from PIL import Image as PILImage
from tqdm import tqdm

PATH_IMAGENES = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\3000imagenes"
PATH_CSV      = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_3000.csv"
PATH_OUTPUT   = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\imagenes-crop-8-3000"

os.makedirs(PATH_OUTPUT, exist_ok=True)

df = pd.read_csv(PATH_CSV)
print(f"Total imágenes: {len(df)}")

Total imágenes: 3000


C:\Users\trodr\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seg_model = xrv.baseline_models.chestx_det.PSPNet()
seg_model.eval()
print("Modelo cargado")

Modelo cargado


In [3]:
def crop_pulmones(img_np, seg_model, margen=0.08):
    # Preprocesar
    img_tensor = xrv.datasets.normalize(img_np, maxval=255, reshape=True)
    img_tensor = torch.from_numpy(img_tensor).float()
    
    if img_tensor.dim() == 3:
        img_tensor = img_tensor.unsqueeze(0)
    
    # Center crop para hacerla cuadrada
    lado = min(img_tensor.shape[2], img_tensor.shape[3])
    img_tensor = T.CenterCrop(lado)(img_tensor)
    
    # Segmentar
    with torch.no_grad():
        output = seg_model(img_tensor)
    
    # Combinar máscaras
    idx_left  = seg_model.targets.index('Left Lung')
    idx_right = seg_model.targets.index('Right Lung')
    mask_lung = np.maximum(
        output[0, idx_left].numpy(),
        output[0, idx_right].numpy()
    )
    mask_binaria = (mask_lung > 0.5).astype(np.uint8)
    
    # Redimensionar máscara a imagen original
    mask_pil = PILImage.fromarray((mask_binaria * 255).astype(np.uint8))
    mask_resized = np.array(
        mask_pil.resize((img_np.shape[1], img_np.shape[0]), PILImage.NEAREST)
    ) // 255
    
    # Bounding box con margen
    filas = np.any(mask_resized, axis=1)
    cols  = np.any(mask_resized, axis=0)
    
    # Si no detecta pulmón, retornar None
    if not np.any(filas) or not np.any(cols):
        return None
    
    y1, y2 = np.where(filas)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    
    h, w = img_np.shape
    margin_y = int((y2 - y1) * margen)
    margin_x = int((x2 - x1) * margen)
    
    y1 = max(0, y1 - margin_y)
    y2 = min(h, y2 + margin_y)
    x1 = max(0, x1 - margin_x)
    x2 = min(w, x2 + margin_x)
    
    return img_np[y1:y2, x1:x2]

In [4]:
errores = []
omitidas = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    ruta_entrada = os.path.join(PATH_IMAGENES, row['full_path'])
    
    # Ruta de salida preservando estructura de carpetas
    ruta_salida = os.path.join(PATH_OUTPUT, row['full_path'])
    os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)
    
    # Saltar si ya existe
    if os.path.exists(ruta_salida):
        continue
    
    try:
        img_pil = PILImage.open(ruta_entrada).convert("L")
        img_np  = np.array(img_pil)
        
        img_crop = crop_pulmones(img_np, seg_model, margen=0.08)
        
        if img_crop is None:
            omitidas.append(row['full_path'])
            continue
        
        # Guardar
        PILImage.fromarray(img_crop.astype(np.uint8)).save(ruta_salida)
        
    except Exception as e:
        errores.append((row['full_path'], str(e)))

print(f"\nListo.")
print(f"  Procesadas: {len(df) - len(errores) - len(omitidas)}")
print(f"  Omitidas (sin pulmón detectado): {len(omitidas)}")
print(f"  Errores: {len(errores)}")

if errores:
    for path, err in errores[:5]:
        print(f"  {path}: {err}")

  0%|          | 0/3000 [00:00<?, ?it/s]

100%|██████████| 3000/3000 [2:01:43<00:00,  2.43s/it]  


Listo.
  Procesadas: 2990
  Omitidas (sin pulmón detectado): 4
  Errores: 6
  files/p16/p16039201/s57846433/f985d27b-5a989257-7fe132f0-e206bd7a-5594d84b.jpg: [Errno 2] No such file or directory: 'C:\\Users\\trodr\\Documents\\proyecto-torax-v2.0\\01-dataset\\dataset-completo-merge\\3000imagenes\\files/p16/p16039201/s57846433/f985d27b-5a989257-7fe132f0-e206bd7a-5594d84b.jpg'
  files/p18/p18863639/s57043164/50e8daed-20523f5b-26c648b8-169695d3-1490317f.jpg: [Errno 2] No such file or directory: 'C:\\Users\\trodr\\Documents\\proyecto-torax-v2.0\\01-dataset\\dataset-completo-merge\\3000imagenes\\files/p18/p18863639/s57043164/50e8daed-20523f5b-26c648b8-169695d3-1490317f.jpg'
  files/p12/p12283084/s55234379/3e82606f-840d78ea-be4290c1-70527e9d-c78aafaf.jpg: [Errno 2] No such file or directory: 'C:\\Users\\trodr\\Documents\\proyecto-torax-v2.0\\01-dataset\\dataset-completo-merge\\3000imagenes\\files/p12/p12283084/s55234379/3e82606f-840d78ea-be4290c1-70527e9d-c78aafaf.jpg'
  files/p19/p19304241/s